## Data Cleaning Summary

We performed the following steps to clean and preprocess the news article data:

1.  **Removed Duplicate Titles**: Articles with identical titles were removed to ensure uniqueness.
2.  **Excluded Irrelevant Data**: Articles containing specific keywords indicating irrelevant content (e.g., photo pages, stock price predictions, advertisements) were filtered out.
3.  **Filtered by Article Length**: Articles that were either extremely long (over 15,000 words) or very short (under 150 words) were removed.
4.  **Standardized Text Case**: All text was converted to lowercase.
5.  **Removed HTML Artifacts**: HTML tags and other web-specific elements were removed from the text.
6.  **Removed Boilerplate Content**: Generic, non-article specific text such as navigation menus, legal disclaimers, social media links, newsletter prompts, and other common website boilerplate were identified and removed using regular expressions.
7.  **Filtered Non-English Articles**: The `langdetect` library was used to identify and keep only articles written in English.
8.  **Applied Keyword Filtering**: The dataset was further refined to include only articles that explicitly mentioned AI-related keywords as whole words (e.g., 'ai', 'machine learning', 'deep learning').
9.  **Lemmatization and Stopword Removal (SpaCy)**: The `clean_text` column was created by processing the `text` column using SpaCy. This involved:
    *   **Lemmatization**: Reducing words to their base form (e.g., 'running' to 'run').
    *   **Stopword Removal**: Eliminating common words that carry little semantic meaning (e.g., 'the', 'is', 'a').
10. **Added Unique Identifier**: A `news_id` column was added by resetting the DataFrame index.

In [ ]:
! pip install langdetect

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from langdetect import detect
import spacy
from tqdm.auto import tqdm

tqdm.pandas()

# Optional for classifier later
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

## 2. Load Raw Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_path = "/content/drive/MyDrive/NLP Final Project/raw_news_data.parquet"

df = pd.read_parquet(df_path)
print("Initial dataset shape:", df.shape)

Initial dataset shape: (199989, 7)


In [ ]:
df.head(10)

,url,date,language,title,text,char_count,word_count
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",3501,483
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,\n\nThis AI video of gymnastics might be the f...,5585,812
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...","\n\nIf using AI feels like a chore, try this -...",5880,884
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,en,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation M...,4072,596
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,en,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers ...,4347,622
5,https://citylife.capetown/lb/uncategorized/how...,2023-12-12,en,Google Releases New Chatbot Bard as a Strong C...,Google Releases New Chatbot Bard as a Strong ...,2634,382
6,https://citylife.capetown/technology/zoom-laun...,2023-09-07,en,Zoom Expands AI Offering with AI Companion and...,Zoom Expands AI Offering with AI Companion an...,3886,552
7,https://citylife.capetown/uncategorized/pro-ai...,2023-08-04,en,Pro-AI Thinking: Enhancing Industrial Environm...,\n\nPro-AI Thinking: Enhancing Industrial Envi...,5094,533
8,https://clickup.com/ai/prompts/business-risk-m...,2024-03-13,en,Best AI Prompts for Business Risk Management,Best AI Prompts for Business Risk ManagementPr...,8328,1040
9,https://crooksandliars.com/2025/12/state-ags-w...,2025-12-15,en,State AGs Warn AI Companies: Clean Up Your Chi...,\nState AGs Warn AI Companies: Clean Up Your C...,7230,1051


## 3. Basic Cleaning

In [ ]:
# 3.1 Remove Duplicates
# In file 01_data_exploration, we found that many articles share the same title and have very high text similarity.
df.drop_duplicates('title', inplace=True)
print("After removing duplicates:", df.shape)

After removing duplicates: (164305, 7)


In [ ]:
# Exclude Irrelevant Data

irrelevant_data = [
    ## photo pages
    "photo rawpixel", "free photo", "premium photo",
    "photo illustration", "ai art photos",

    ## stock price
    'price prediction price', '2025 2026 2030',

    ## advertisement
    "product hunt",

    ## homepage
    "daily mail",
    "press release",
    "public radio"
]

# Define a cleaning function for titles for exclusion matching
def clean_string_for_pattern_matching(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text) # Remove punctuation and special characters, keep alphanumeric and spaces
    return text

# Apply this cleaning to the 'title' column before checking for irrelevant data
# Create a temporary series of cleaned titles for the purpose of filtering
cleaned_titles_for_exclusion = df['title'].apply(clean_string_for_pattern_matching)

# Now use this cleaned series for filtering
exclude_pattern_for_cleaned_titles = '|'.join(irrelevant_data)

df = df[~cleaned_titles_for_exclusion.str.contains(exclude_pattern_for_cleaned_titles, regex=True)]
print("After removing irrelevant data based on cleaned titles:", df.shape)

After removing irrelevant data based on cleaned titles: (153596, 7)


In [ ]:
# 3.4 Remove Very Long Articles
df = df[df['word_count'] < 15000] # filter >15000 words
print("After removing long articles (>15000 words):", df.shape)

After removing long articles (>15000 words): (153585, 7)


In [ ]:
# 3.4 Remove Very Short Articles
df = df[df['word_count'] > 150]  # filter <150 words
print("After removing short articles (<150 words):", df.shape)

After removing short articles (<150 words): (151447, 7)


In [ ]:
# 3.2 Lowercase
df['text'] = df['text'].str.lower()

In [ ]:
# 3.3 Remove HTML Artifacts
df['text'] = df['text'].str.replace(r'<.*?>', ' ', regex=True)

In [ ]:
df.head(10)

,url,date,language,title,text,char_count,word_count
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","bad idea ai price (bad), market cap, price tod...",3501,483
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,\n\nthis ai video of gymnastics might be the f...,5585,812
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...","\n\nif using ai feels like a chore, try this -...",5880,884
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,en,The Road Ahead: How China's AI Foundation Mode...,the road ahead: how china's ai foundation m...,4072,596
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,en,Microsoft and Nvidia to Empower Developers wit...,microsoft and nvidia to empower developers ...,4347,622
5,https://citylife.capetown/lb/uncategorized/how...,2023-12-12,en,Google Releases New Chatbot Bard as a Strong C...,google releases new chatbot bard as a strong ...,2634,382
6,https://citylife.capetown/technology/zoom-laun...,2023-09-07,en,Zoom Expands AI Offering with AI Companion and...,zoom expands ai offering with ai companion an...,3886,552
7,https://citylife.capetown/uncategorized/pro-ai...,2023-08-04,en,Pro-AI Thinking: Enhancing Industrial Environm...,\n\npro-ai thinking: enhancing industrial envi...,5094,533
8,https://clickup.com/ai/prompts/business-risk-m...,2024-03-13,en,Best AI Prompts for Business Risk Management,best ai prompts for business risk managementpr...,8328,1040
9,https://crooksandliars.com/2025/12/state-ags-w...,2025-12-15,en,State AGs Warn AI Companies: Clean Up Your Chi...,\nstate ags warn ai companies: clean up your c...,7230,1051


### 4.3 Remove More Generic Boilerplate and Repeated Titles

In [ ]:
import re

########################################
# 2. Remove Strong Boilerplate Patterns
########################################

boilerplate_patterns = [

    # Navigation
    r'\bmenu\b',
    r'\bsearch\b',
    r'\bstore\b',
    r'\bhome\b',
    r'\bsections\b',
    r'\bsitemap\b',

    # Legal / cookies
    r'privacy policy',
    r'terms of service',
    r'all rights reserved',
    r'cookie[s]?',
    r'analytics trackers',
    r'affiliate advertising program',

    # Social media
    r'follow us',
    r'facebook',
    r'twitter',
    r'instagram',
    r'mastodon',
    r'flipboard',
    r'bluesky',

    # Newsletter
    r'newsletter',
    r'subscribe',
    r'sign up',
    r'log in',

    # Ads / promotions
    r'tl;dr',
    r'on sale',
    r'lifetime subscription',
    r'no credit card',
    r'reg\.',
    r'\$\d+',

    # Related posts
    r'read the rest',
    r'recent posts',
    r'related post',
    r'you missed',

    # FAQ blocks
    r'\bfaq\b',
    r'\bq:\b',
    r'\ba:\b',

    # Crypto price pages
    r'market cap',
    r'circulating supply',
    r'24h volume',
    r'trading volume',
    r'market data',
    r'price today',

    # Footer junk
    r'contact us',
    r'about us',
    r'advertise',
    r'careers',
    r'trust & ethics',

    # Multilanguage navigation
    r'idiomas',
    r'languages',
    r'contacto',
    r'контакт',

]

full_pattern = '|'.join(boilerplate_patterns)

df['text'] = df['text'].progress_apply(
    lambda x: re.sub(full_pattern, ' ', x, flags=re.IGNORECASE)
)


########################################
# 3. Remove Long Junk Sections
########################################

# Remove "recent posts" sections
df['text'] = df['text'].progress_apply(
    lambda x: re.sub(r'recent posts.*', ' ', x, flags=re.IGNORECASE)
)

# Remove footer blocks
df['text'] = df['text'].progress_apply(
    lambda x: re.sub(r'about us.*', ' ', x, flags=re.IGNORECASE)
)

# Remove cookie/legal blocks
df['text'] = df['text'].progress_apply(
    lambda x: re.sub(r'we use cookies.*', ' ', x, flags=re.IGNORECASE)
)


########################################
# 4. Remove Excess Whitespace
########################################

df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True)
df['text'] = df['text'].str.strip()


########################################
# 5. Remove Extremely Noisy Articles
########################################

# Remove texts that are too short after cleaning
df = df[df['text'].str.len() > 500]


print("After strong boilerplate cleaning:", df.shape)

  0%|          | 0/151447 [00:00<?, ?it/s]

  0%|          | 0/151447 [00:00<?, ?it/s]

  0%|          | 0/151447 [00:00<?, ?it/s]

  0%|          | 0/151447 [00:00<?, ?it/s]

After strong boilerplate cleaning: (151447, 7)


In [ ]:
# 3.5 Keep Only English Articles
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

df['is_english'] = df['text'].progress_apply(is_english)
df = df[df['is_english']]
print("After keeping only English articles:", df.shape)

  0%|          | 0/151447 [00:00<?, ?it/s]

After keeping only English articles: (151279, 8)


## 4. Rule-Based AI-Relevant Filtering

In [ ]:
# Aggressive keyword filtering was not applied because this dataset was already pre-filtered to include AI-related articles.

import re

# 4.1 Keywords
keywords = [
    'ai', 'machine learning', 'automation', 'robotics', 'artificial intelligence',
    'generative ai', 'deep learning', 'large language model', 'neural network'
]

# Build a regex pattern to match each keyword as a whole word
regex_keywords = [r'\b' + re.escape(keyword) + r'\b' for keyword in keywords]
pattern = '|'.join(regex_keywords)

df = df[df['text'].str.contains(pattern, case=False, na=False, regex=True)]
print("After keyword filtering (whole words only):", df.shape)


After keyword filtering (whole words only): (146876, 8)


In [ ]:
# 4.1 Keywords
"""
keywords = [
    'ai', 'machine learning', 'automation', 'robotics', 'artificial intelligence',
    'generative ai', 'deep learning', 'llm', 'neural network'
]
pattern = '|'.join(keywords)
df = df[df['text'].str.contains(pattern)]
print("After keyword filtering:", df.shape)
"""

'\nkeywords = [\n    \'ai\', \'machine learning\', \'automation\', \'robotics\', \'artificial intelligence\',\n    \'generative ai\', \'deep learning\', \'llm\', \'neural network\'\n]\npattern = \'|\'.join(keywords)\ndf = df[df[\'text\'].str.contains(pattern)]\nprint("After keyword filtering:", df.shape)\n'

## 5. SpaCy Lemmatization & Stopword Removal

In [ ]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 61.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# 5.1 Load SpaCy Model
nlp = spacy.load("en_core_web_md", disable=["ner","parser"])

# 5.2 Preprocessing Function
def preprocess_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]
    return " ".join(tokens)

# 5.3 Apply Lemmatization
df['clean_text'] = df['text'].progress_apply(preprocess_text)
print("Added clean_text column with lemmatized content.")

  0%|          | 0/146876 [00:00<?, ?it/s]

Added clean_text column with lemmatized content.


In [ ]:
df[['text','clean_text']].sample(3)

,text,clean_text
122022,saline township residents express anger over n...,saline township resident express anger new ai ...
137190,"emirati, international firms showcase new ai s...",emirati international firm showcase new ai sol...
184476,"odisha bse gears up for matric exams, cctvs an...",odisha bse gear matric exam cctv ai camera ins...


In [ ]:
df.reset_index(inplace=True)
df.rename(columns={'index': 'news_id'}, inplace=True)
print("Added 'news_id' column to DataFrame.")
df.head()

Added 'news_id' column to DataFrame.


,news_id,url,date,language,title,text,char_count,word_count,is_english,clean_text
0,0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","bad idea ai price (bad), , & chart history - b...",3501,483,True,bad idea ai price bad chart history blockworks...
1,1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,this ai video of gymnastics might be the freak...,5585,812,True,ai video gymnastic freaky see boe boe blog pos...
2,2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...","if using ai feels like a chore, try this - boi...",5880,884,True,ai feel like chore try boe boe blog post forum...
3,3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,en,The Road Ahead: How China's AI Foundation Mode...,the road ahead: how china's ai foundation mode...,4072,596,True,road ahead china ai foundation model shape fut...
4,4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,en,Microsoft and Nvidia to Empower Developers wit...,microsoft and nvidia to empower developers wit...,4347,622,True,microsoft nvidia empower developer window ai s...


In [ ]:
df.shape

(146876, 10)

In [ ]:
df.to_parquet("/content/drive/MyDrive/NLP Final Project/clean_news_data.parquet", index=False)
print("Saved cleaned dataset with clean_text column for Topic Modeling.")

Saved cleaned dataset with clean_text column for Topic Modeling.


## 6. Optional Classifier Filtering (Advanced)

In [ ]:
# labeled_df = pd.read_csv("labeled_articles.csv")  # contains 'text' and 'label' (1=relevant,0=not)
# X = labeled_df['text']
# y = labeled_df['label']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
# X_train_vec = vectorizer.fit_transform(X_train)
# X_test_vec = vectorizer.transform(X_test)

# clf = LogisticRegression(max_iter=500)
# clf.fit(X_train_vec, y_train)
# y_pred = clf.predict(X_test_vec)
# print(classification_report(y_test, y_pred))

# Apply classifier to full dataset
# X_full_vec = vectorizer.transform(df['clean_text'])
# df['pred_relevant'] = clf.predict(X_full_vec)
# df = df[df['pred_relevant'] == 1]